In [ ]:
import re

# === Analizador Léxico ===
# Definición de patrones para diferentes tipos de tokens
token_patron = {
    "KEYWORDS": r'\b(if|else|while|return|int|float|void|then|print|printf)\b',
    "IDENTIFIER" : r'\b[a-zA-Z][a-zA-Z0-9]*\b',
    "NUMBER" : r'\b\d+(\.\d+)?\b',
    "OPERATORS" : r'[+\-*/=<>!]',
    "DELIMITERS" : r'[(),;{}]',
    "WHITESPACE" : r'\s+'
}

def identificar_tokens(texto):
    # Unimos todos los patrones en un único patrón usando grupos nombrados
    patron_general = '|'.join(f'(?P<{token}>{patron})' for token, patron in token_patron.items())
    patron_regex = re.compile(patron_general)

    tokens_encontrados = []
    for match in patron_regex.finditer(texto):
        for token, valor in match.groupdict().items():
            if valor is not None and token != "WHITESPACE": # Ignoramos espacios en blanco
                tokens_encontrados.append((token, valor))
    return tokens_encontrados

In [ ]:
class NodoAST:
    # Clase para todos los nodos del AST
    pass

    def traducirPy(self):
        # Traduccion de C++ a Python
        raise NotImplementedError("Método traducirPy() no implementado en este Nodo")
    
    def traducirJava(self):
        # Traduccion de C++ a Java
        raise NotImplementedError("Método traducirJava() no implementado en este Nodo")
    
    def generarCodigo():
        # Traduccion de C++ a Assembler
        raise NotImplementedError("Método generarCodigo() no implementado en este Nodo")

# Nodo Programa
class NodoPrograma(NodoAST):
    # Nodo que representa a un programa completo
    def __init__(self, funciones, main):
        self.variables = []
        self.funciones = funciones
        self.main = main

    def traducirPy(self):
        funciones = "\n\n".join(f.traducirPy() for f in self.funciones)
        main = self.main.traducirPy() if self.main else ""
        return f"{funciones}\n\n{main}".strip()

    def traducirJava(self):
        funciones = "\n\n".join(f.traducirJava() for f in self.funciones)
        main = self.main.traducirJava() if self.main else ""
        return f"{funciones}\n\n{main}".strip()

    def generarCodigo(self):
        funciones = "\n\n".join(f.generarCodigo() for f in self.funciones)
        main = self.main.generarCodigo() if self.main else ""
        return f"{funciones}\n\n{main}".strip()

    def optimizar(self):
        funciones_opt = [f.optimizar() for f in self.funciones]
        main_opt = self.main.optimizar() if self.main else None
        return NodoPrograma(funciones_opt, main_opt)

class NodoFuncion(NodoAST):
    # Nodo que representa una función
    def __init__(self, nombre, parametros, cuerpo):
        self.nombre = nombre
        self.parametros = parametros
        self.cuerpo = cuerpo

    # Traducción a Python
    def traducirPy(self):
        params = ', '.join(p.traducirPy() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirPy() for c in self.cuerpo)
        return f"def {self.nombre[1]}({params}):\n    {cuerpo}"
    
    # Traducción a Java
    def traducirJava(self):
        params = ', '.join(p.traducirJava() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirJava() for c in self.cuerpo)
        return f"public class Programa {{\n    public static int {self.nombre[1]}({params}) {{\n        {cuerpo}\n    }}\n}}"
    
    def generarCodigo(self):
        cuerpo = "\n    ".join(c.generarCodigo() for c in self.cuerpo)
        return f"; Función {self.nombre[1]}\n{self.nombre[1]}:\n    {cuerpo}"
    
    def optimizar(self):
        cuerpo_opt = [c.optimizar() for c in self.cuerpo]
        return NodoFuncion(self.nombre, self.parametros, cuerpo_opt)

class NodoParametro(NodoAST):
    # Nodo que representa un parámetro de función
    def __init__(self, nombre, tipo):
        self.nombre = nombre
        self.tipo = tipo

    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return f"{self.tipo[1]} {self.nombre[1]}"
    
    def generarCodigo(self):
        return f"; param {self.nombre[1]}"

class NodoAsignacion(NodoAST):
    # Nodo que representa una asignación
    def __init__(self, nombre, expresion):
        self.nombre = nombre
        self.expresion = expresion

    def traducirPy(self):
        return f"{self.nombre[1]} = {self.expresion.traducirPy()}"
    
    def traducirJava(self):
        return f"{self.nombre[1]} = {self.expresion.traducirJava()};"
    
    def generarCodigo(self):
        return f"MOV {self.nombre[1]}, {self.expresion.generarCodigo()}"
    
    def optimizar(self):
        return NodoAsignacion(self.nombre, self.expresion.optimizar())

class NodoOperacion(NodoAST):
    # Nodo que representa una operación aritmética
    def __init__(self, izquierda, operador, derecha):
        self.izquierda = izquierda
        self.operador = operador
        self.derecha = derecha

    def traducirPy(self):
        return f"({self.izquierda.traducirPy()} {self.operador[1]} {self.derecha.traducirPy()})"
    
    def traducirJava(self):
        return f"({self.izquierda.traducirJava()} {self.operador[1]} {self.derecha.traducirJava()})"
    
    def generarCodigo(self):
        izq = self.izquierda.generarCodigo()
        der = self.derecha.generarCodigo()
        op_map = {'+': 'ADD', '-': 'SUB', '*': 'MUL', '/': 'DIV'}
        op = op_map.get(self.operador[1], self.operador[1])
        return f"{op} {izq}, {der}"
    
    def optimizar(self):
        izq = self.izquierda.optimizar()
        der = self.derecha.optimizar()
        # Si ambos lados son números, resuelve la operación
        if isinstance(izq, NodoNumero) and isinstance(der, NodoNumero):
            op = self.operador[1]
            vi = float(izq.valor[1])
            vd = float(der.valor[1])
            if op == '+': resultado = vi + vd
            elif op == '-': resultado = vi - vd
            elif op == '*': resultado = vi * vd
            elif op == '/': resultado = vi / vd
            return NodoNumero(('NUMBER', str(resultado)))
        return NodoOperacion(izq, self.operador, der)

class NodoRetorno(NodoAST):
    # Nodo que representa la sentencia de return
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f"return {self.expresion.traducirPy()}"
    
    def traducirJava(self):
        return f"return {self.expresion.traducirJava()};"
    
    def generarCodigo(self):
        return f"MOV eax, {self.expresion.generarCodigo()}\n    RET"
    
    def optimizar(self):
        return NodoRetorno(self.expresion.optimizar())

class NodoIdentificador(NodoAST):
    # Nodo que representa a un identificador
    def __init__(self, nombre):
        self.nombre = nombre

    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return self.nombre[1]
    
    def generarCodigo(self):
        return self.nombre[1]
    
    def optimizar(self):
        return self  # una variable no se puede optimizar

class NodoNumero(NodoAST):
    # Nodo que representa un número
    def __init__(self, valor):
        self.valor = valor

    def traducirPy(self):
        return str(self.valor[1])
    
    def traducirJava(self):
        return str(self.valor[1])
    
    def generarCodigo(self):
        return str(self.valor[1])
    
    def optimizar(self):
        return self  # un número ya está optimizado

class NodoLlamadaFuncion(NodoAST):
    # Nodo que representa una llamada a función
    def __init__(self, nombref, argumentos):
        self.nombre_funcion = nombref
        self.argumentos = argumentos

class NodoPrint(NodoAST):
    # Nodo que representa una sentencia de impresión
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f"print({self.expresion.traducirPy()})"
    
    def traducirJava(self):
        return f"System.out.println({self.expresion.traducirJava()});"
    
    def generarCodigo(self):
        return f"PRINT {self.expresion.generarCodigo()}"
    
    def optimizar(self):
        return NodoPrint(self.expresion.optimizar())
    
class NodoPrintf(NodoAST):
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f"print({self.expresion.traducirPy()})"

    def traducirJava(self):
        return f"System.out.printf({self.expresion.traducirJava()});"

    def generarCodigo(self):
        return f"PRINT {self.expresion.generarCodigo()}"
    
    def optimizar(self):
        return NodoPrintf(self.expresion.optimizar())
    
class NodoFloat(NodoAST):
    def __init__(self, valor):
        self.valor = valor

    def traducirPy(self):
        return str(self.valor[1])

    def traducirJava(self):
        return f"{self.valor[1]}f"

    def generarCodigo(self):
        return str(self.valor[1])

    def optimizar(self):
        return self

In [ ]:
class TablaSimbolos:
    def __init__(self):
        self.simbolos = {}  # nombre -> {tipo, valor}

    def agregar(self, nombre, tipo, valor=None):
        if nombre in self.simbolos:
            raise Exception(f"Error semántico: la variable '{nombre}' ya fue declarada")
        self.simbolos[nombre] = {'tipo': tipo, 'valor': valor}

    def buscar(self, nombre):
        if nombre not in self.simbolos:
            raise Exception(f"Error semántico: la variable '{nombre}' no fue declarada")
        return self.simbolos[nombre]

    def actualizar(self, nombre, valor):
        self.buscar(nombre)  # verifica que exista
        self.simbolos[nombre]['valor'] = valor

    def imprimir(self):
        print("\n=== Tabla de Símbolos ===")
        for nombre, info in self.simbolos.items():
            print(f"  {nombre}: tipo={info['tipo']}, valor={info['valor']}")
        print("========================\n")


class AnalizadorSemantico:
    def __init__(self):
        self.tabla = TablaSimbolos()
        self.funciones = {}  # nombre -> {params, tipo_retorno}

    def analizar(self, nodo):
        if isinstance(nodo, NodoPrograma):
            for funcion in nodo.funciones:
                self.analizar(funcion)
            self.analizar(nodo.main)

        elif isinstance(nodo, NodoFuncion):
            # Registrar la función
            self.funciones[nodo.nombre[1]] = {
                'params': nodo.parametros,
                'tipo_retorno': 'int'
            }
            # Registrar parámetros en la tabla
            for param in nodo.parametros:
                self.tabla.agregar(param.nombre[1], param.tipo[1])
            # Analizar el cuerpo
            for instruccion in nodo.cuerpo:
                self.analizar(instruccion)

        elif isinstance(nodo, NodoAsignacion):
            # Registrar o actualizar la variable
            tipo = 'float' if isinstance(nodo.expresion, NodoFloat) else 'int'
            try:
                self.tabla.buscar(nodo.nombre[1])
                self.tabla.actualizar(nodo.nombre[1], None)
            except:
                self.tabla.agregar(nodo.nombre[1], tipo)
            self.analizar(nodo.expresion)

        elif isinstance(nodo, NodoOperacion):
            # Verificar división entre cero
            if nodo.operador[1] == '/' and isinstance(nodo.derecha, NodoNumero):
                if float(nodo.derecha.valor[1]) == 0:
                    raise Exception("Error semántico: división entre cero detectada")
            self.analizar(nodo.izquierda)
            self.analizar(nodo.derecha)

        elif isinstance(nodo, NodoIdentificador):
            # Verificar que la variable fue declarada
            self.tabla.buscar(nodo.nombre[1])

        elif isinstance(nodo, NodoRetorno):
            self.analizar(nodo.expresion)

        elif isinstance(nodo, NodoPrint) or isinstance(nodo, NodoPrintf):
            self.analizar(nodo.expresion)

        elif isinstance(nodo, NodoLlamadaFuncion):
            # Verificar que la función existe
            if nodo.nombre_funcion not in self.funciones:
                raise Exception(f"Error semántico: función '{nodo.nombre_funcion}' no declarada")

        # NodoNumero y NodoFloat no necesitan análisis

In [ ]:
# === Análisis sintáctico ===
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def obtener_token_actual(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None
    
    def coincidir(self, tipo_esperado):
        token_actual = self.obtener_token_actual()
        if token_actual and token_actual[0] == tipo_esperado:
            self.pos += 1
            return token_actual
        raise SyntaxError(f"Error sintáctico: se esperaba {tipo_esperado} pero se encontró: {token_actual}")

    def parse(self):
        # Punto de entrada: se espera una función
        return self.programa()

    def programa(self):
        funciones = []
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != 'main':
            funciones.append(self.funcion())
        
        # main es opcional
        if self.obtener_token_actual() and self.obtener_token_actual()[1] == 'main':
            main = self.funcion()
        else:
            main = None
        
        return NodoPrograma(funciones, main)

    def funcion(self):
        self.coincidir("KEYWORDS")        # Tipo de retorno 'int'
        nombre = self.coincidir("IDENTIFIER")  # Nombre de la función
        self.coincidir("DELIMITERS")      # '('
        if nombre[1] == 'main':
            parametros = []  # main no tiene parámetros
        else:
            parametros = self.parametros()    # ← capturar la lista
        self.coincidir("DELIMITERS")      # ')'
        self.coincidir("DELIMITERS")      # '{'
        cuerpo = self.cuerpo()            # ← capturar las instrucciones
        self.coincidir("DELIMITERS")      # '}'
        return NodoFuncion(nombre, parametros, cuerpo)  # ← retornar el nodo

    def parametros(self): # Alcance privado colocar __
        # Reglas para parámetros: int IDENTIFIER (, int IDENTIFIER)*
        lista_parametros = []
        tipo = self.coincidir("KEYWORDS")  # Tipo de parámetro 'int'
        nombre = self.coincidir("IDENTIFIER")  # Nombre del parámetro
        lista_parametros.append(NodoParametro(nombre, tipo))
        while self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
            self.coincidir("DELIMITERS")  # Coma ','
            tipo = self.coincidir("KEYWORDS")  # Tipo de parámetro 'int'
            nombre = self.coincidir("IDENTIFIER")  # Nombre del parámetro
            lista_parametros.append(NodoParametro(nombre, tipo))
        return lista_parametros
    
    def cuerpo(self):
        # Gramática para el cuerpo: return IDENTIFIER OPERATOR IDENTIFIER ;
        instrucciones = []
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != '}':
            if self.obtener_token_actual()[1] == 'return':
                instrucciones.append(self.retorno())
            elif self.obtener_token_actual()[1] == 'print':
                instrucciones.append(self.sentencia_print())
            elif self.obtener_token_actual()[1] == 'printf':  # ← nuevo
                instrucciones.append(self.sentencia_printf())
            else:
                instrucciones.append(self.asignacion())
        return instrucciones
        
    def sentencia_printf(self):
        self.coincidir("KEYWORDS")   # 'printf'
        self.coincidir("DELIMITERS") # '('
        expresion = self.expresion()
        self.coincidir("DELIMITERS") # ')'
        self.coincidir("DELIMITERS") # ';'
        return NodoPrintf(expresion)
    
    def sentencia_print(self):
        self.coincidir("KEYWORDS")   # 'print'
        self.coincidir("DELIMITERS") # '('
        expresion = self.expresion()
        self.coincidir("DELIMITERS") # ')'
        self.coincidir("DELIMITERS") # ';'
        return NodoPrint(expresion)

    def asignacion(self):
        self.coincidir("KEYWORDS")             # Tipo 'int' o 'float'
        nombre = self.coincidir("IDENTIFIER")  # Nombre de la variable
        self.coincidir("OPERATORS")            # '='
        expresion = self.expresion()           # Lado derecho
        self.coincidir("DELIMITERS")           # ';'
        return NodoAsignacion(nombre, expresion)

    def retorno(self):
        self.coincidir("KEYWORDS")  # 'return'
        expresion = self.expresion()
        self.coincidir("DELIMITERS")  # Punto y coma ';'
        return NodoRetorno(expresion)
    
    def expresion(self):
        izquierda = self.termino()
        while self.obtener_token_actual() and self.obtener_token_actual()[0] == 'OPERATORS':
            operador = self.coincidir("OPERATORS")
            derecha = self.termino()
            izquierda = NodoOperacion(izquierda, operador, derecha)
        return izquierda
    
    def termino(self):
        token = self.obtener_token_actual()
        if token[0] == 'NUMBER':
            numero = self.coincidir("NUMBER")
            if '.' in numero[1]:  # ← tiene punto decimal?
                return NodoFloat(numero)
            return NodoNumero(numero)
        elif token[0] == 'IDENTIFIER':
            identificador = self.coincidir("IDENTIFIER")
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == '(':
                self.coincidir("DELIMITERS")  # Paréntesis de apertura '('
                argumentos = self.llamadaFuncion()
                self.coincidir("DELIMITERS")  # Paréntesis de cierre ')'
                return NodoLlamadaFuncion(identificador[1], argumentos)
            else:
                return NodoIdentificador(identificador)
        else:
            raise SyntaxError(f"Expresión no válida: {token}")

    def llamadaFuncion(self):
        argumentos = []
        # Reglas para argumentos: IDENTIFIER | NUMBER (, IDENTIFIER | NUMBER)*
        sigue = True
        token = self.obtener_token_actual()
        while sigue:
            sigue = False
            if token[0] == 'NUMBER':
                argumento = NodoNumero(self.coincidir("NUMBER"))
            elif token[0] == 'IDENTIFIER':
                argumento = NodoIdentificador(self.coincidir("IDENTIFIER"))
            else:
                raise SyntaxError(f"Error de sintaxis, se esperaba un IDENTIFICADOR|NUMERO pero se encontró: {token}")
            argumentos.append(argumento)
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
                self.coincidir("DELIMITERS")  # Coma ','
                token = self.obtener_token_actual()
                sigue = True
        return argumentos
        


In [ ]:
import json

def imprimir_ast(nodo):
    if isinstance(nodo, NodoFuncion):
        return {
            'Funcion': nodo.nombre,
            'Parametros': [imprimir_ast(p) for p in nodo.parametros],
            'Cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]
        }
    elif isinstance(nodo, NodoParametro):
        return {
            'Parametro': nodo.nombre,
            'Tipo': nodo.tipo
        }
    elif isinstance(nodo, NodoAsignacion):
        return {
            'Asignacion': nodo.nombre,
            'Expresion': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            'Operacion': nodo.operador,
            'Izquierda': imprimir_ast(nodo.izquierda),
            'Derecha': imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoRetorno):
        return {
            'Return': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoIdentificador):
        return {
            'Identificador': nodo.nombre
        }
    elif isinstance(nodo, NodoNumero):
        return {
            'Numero': nodo.valor
        }
    elif isinstance(nodo, NodoPrint):
        return {
            'Print': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoPrograma):
        return {
            'Programa': {
                'Funciones': [imprimir_ast(f) for f in nodo.funciones],
                'Main': imprimir_ast(nodo.main)
            }
        }
    elif isinstance(nodo, NodoFloat):
        return {
            'Float': nodo.valor
        }
    else: 
        return {}

In [ ]:
codigo_fuente = """
int suma(int a, int b) {
    int resultado = 3.14 + 2.5;
    return resultado;
}
"""
tokens = identificar_tokens(codigo_fuente)

try:
    print("\nIniciando análisis sintáctico...")
    parser = Parser(tokens)
    arbol_ast = parser.parse()
    print("Análisis sintáctico completado con éxito.")
except SyntaxError as e:
    print(e)
    arbol_ast = None

if arbol_ast:
    print(json.dumps(imprimir_ast(arbol_ast), indent=1))
    print(arbol_ast.traducirPy())
    print(arbol_ast.traducirJava())

In [ ]:
nodoExp = NodoOperacion(
    NodoNumero(('NUMBER', '3')),
    ('OPERATORS', '+'),
    NodoNumero(('NUMBER', '5'))
)
exp_opt = nodoExp.optimizar()  # ← nombre correcto
print(json.dumps(imprimir_ast(exp_opt), indent=1))

In [ ]:
print("Tokens encontrados:")
for tipo, valor in tokens:
    print(f"{tipo}: {valor}")

In [ ]:
codigo_fuente_main = """
int suma(int a, int b) {
    int resultado = a + b;
    print(resultado);
    return resultado;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}
"""
tokens_nuevo = identificar_tokens(codigo_fuente_main)

try:
    print("\nIniciando análisis sintáctico main...")
    parser_main = Parser(tokens_nuevo)
    arbol_ast_main = parser_main.parse()
    print("Análisis sintáctico completado con éxito.")
except SyntaxError as e:
    print(e)
    arbol_ast_main = None

In [ ]:
if arbol_ast_main:
    print(json.dumps(imprimir_ast(arbol_ast_main), indent=1))
    try:
        print("\nIniciando análisis semántico...")
        semantico = AnalizadorSemantico()
        semantico.analizar(arbol_ast_main)  # ← ahora sí existe
        semantico.tabla.imprimir()
        print("Análisis semántico completado con éxito.")
    except Exception as e:
        print(e)

In [ ]:
print(json.dumps(imprimir_ast(arbol_ast_main), indent=1))

In [ ]:
import subprocess

In [ ]:
def compilar(programa):
    # Generar el código en ensamblador
    codigo_asm = programa.generarCodigo()
    print(codigo_asm)
    text_file = open("ejasmcompiladores.asm", "w")
    text_file.write(codigo_asm)
    text_file.close()
    subprocess.run(["nasm", "-f", "elf64", "ejasmcompiladores.asm"])
    subprocess.run(["ld", "-m", "elf_i386", "ejasmcompiladores.o", "-o", "ejasmcompiladores"])
    # Falta agregar el resultado como variable